# Late Fusion Inference Pipeline

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────────
!pip install -q huggingface_hub

In [ ]:
import os, pickle, json
import numpy as np
import cv2
from PIL import Image
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input as prep_mammo
from tensorflow.keras.applications.resnet50 import preprocess_input as prep_us
from huggingface_hub import hf_hub_download
import warnings
warnings.filterwarnings('ignore')

print('TF:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

## Load models from Hugging face

In [ ]:
from huggingface_hub import hf_hub_download

# ── Hugging Face repo ──────────────────────────────────────────────────────────
HF_REPO = 'hab200/breast-cancer-late-fusion' 

print('Downloading models from Hugging Face...')

# Mammogram model (EfficientNetV2L fine-tuned on CBIS-DDSM)
mammo_path = hf_hub_download(repo_id=HF_REPO, filename='Models/mammo_ft_classification.keras')

# Ultrasound model (ResNet50 fine-tuned on BUS-BRA)
us_path = hf_hub_download(repo_id=HF_REPO, filename='Models/ultrasound_ft_classification.keras')

# Fusion model (Logistic Regression)
fusion_path = hf_hub_download(repo_id=HF_REPO, filename='Models/fusion_model.pkl')

# Fusion config (threshold + metadata)
config_path = hf_hub_download(repo_id=HF_REPO, filename='Models/fusion_config.json')

print('All models downloaded ')

In [ ]:
# ── Load Models ────────────────────────────────────────────────────────────────
print('Loading models...')

mammo_model = load_model(mammo_path, compile=False)
us_model = load_model(us_path, compile=False)

with open(fusion_path, 'rb') as f:
 fusion_model = pickle.load(f)

with open(config_path) as f:
 config = json.load(f)

THRESHOLD = config['threshold'] # 0.3341
CLASS_NAMES = config['class_names'] # ['benign', 'malignant']

print(f'Mammogram model input: {mammo_model.input_shape}')
print(f'Ultrasound model input: {us_model.input_shape}')
print(f'Fusion threshold: {THRESHOLD}')
print('Models loaded ')

## 2. Preprocessing Functions

In [ ]:
def preprocess_mammogram(img_path: str) -> np.ndarray:
 """
 Mammogram preprocessing:
 - BGR → RGB
 - Resize to 456×456
 - CLAHE (Contrast Limited Adaptive Histogram Equalization)
 - EfficientNetV2L preprocess_input
 """
 img = cv2.imread(img_path)
 if img is None:
 raise ValueError(f'Cannot read image: {img_path}')
 img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
 img = cv2.resize(img, (456, 456), interpolation=cv2.INTER_CUBIC)

 # CLAHE
 gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
 clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
 eq = clahe.apply(gray)
 img = cv2.cvtColor(eq, cv2.COLOR_GRAY2RGB)

 img = img.astype(np.float32)
 img = prep_mammo(img)
 return np.expand_dims(img, axis=0) # (1, 456, 456, 3)


def preprocess_ultrasound(img_path: str) -> np.ndarray:
 """
 Ultrasound preprocessing:
 - PIL open → RGB
 - Resize to 224×224
 - ResNet50 preprocess_input
 """
 img = Image.open(img_path).convert('RGB')
 img = img.resize((224, 224))
 arr = np.array(img, dtype=np.float32)
 arr = prep_us(arr)
 return np.expand_dims(arr, axis=0) # (1, 224, 224, 3)


print('Preprocessing functions ready ')

## 3. Inference Function

In [ ]:
def predict(
 mammo_img_path: str,
 us_img_path: str,
 threshold: float = THRESHOLD
) -> dict:
 """
 Late Fusion Inference — Mammogram + Ultrasound.
 """
 # ── 1. Preprocess ──────────────────────────────────────────────────────────
 mammo_arr = preprocess_mammogram(mammo_img_path)
 us_arr = preprocess_ultrasound(us_img_path)

 # ── 2. Individual model inference ─────────────────────────────────────────
 mammo_probs = mammo_model.predict(mammo_arr, verbose=0)[0] # (1,) or (2,)
 us_probs = us_model.predict(us_arr, verbose=0)[0] # (1,) or (2,)

 ps_mammo = float(mammo_probs[0]) if len(mammo_probs) == 1 else float(mammo_probs[1])
 ps_us = float(us_probs[0]) if len(us_probs) == 1 else float(us_probs[1])

 # ── 3. Late fusion ─────────────────────────────────────────────────────────
 import pandas as pd
 features = pd.DataFrame([[ps_mammo, ps_us]], columns=['PS_mammo_img', 'PS_us_img'])
 fusion_score = float(fusion_model.predict_proba(features)[0][1])

 # ── 4. Decision ───────────────────────────────────────────────────────────
 if fusion_score >= threshold:
 prediction = 'malignant'
 confidence = 0.5 + 0.5 * ((fusion_score - threshold) / (1.0 - threshold))
 else:
 prediction = 'benign'
 confidence = 0.5 + 0.5 * ((threshold - fusion_score) / threshold)

 return {
 'class': prediction,
 'confidence': round(confidence * 100, 2)
 }
print('predict() function updated for Backend ')

## 4. Test 

In [ ]:
# ── Test ──────────────────────────────────────────────────────────────────────
MAMMO_TEST_IMG = '/kaggle/input/datasets/awsaf49/cbis-ddsm-breast-cancer-image-dataset/jpeg/1.3.6.1.4.1.9590.100.1.2.38716633512694080011766947013857405484/1-192.jpg' 
US_TEST_IMG = '/kaggle/input/datasets/orvile/bus-bra-a-breast-ultrasound-dataset/BUSBRA/BUSBRA/Images/bus_0588-r.png' 

result = predict(MAMMO_TEST_IMG, US_TEST_IMG)

print('=' * 45)
print('PREDICTION RESULT')
print('=' * 45)
for k, v in result.items():
 print(f' {k:<20}: {v}')
print('=' * 45)